# 02 · Linear regression and gradient descent

After this notebook you can write down the linear model and its loss, derive the gradient at a whiteboard, solve it two ways in NumPy (normal equation and gradient descent), read a loss curve to fix a learning rate, and explain why feature scaling and mini-batches matter.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · ML workflow](01_ml_workflow_and_train_test_split.ipynb), [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb)

## Why this matters in interviews

- Gradient descent is how every neural network, every LLM included, is trained. Linear regression is the smallest model where you can see the whole loop: **model → loss → gradient → update**.
- "Derive the gradient of MSE" and "closed form or gradient descent, when would you use which?" are standard whiteboard questions.
- Loss-curve reading transfers straight to fine-tuning: a curve that diverges, crawls or bounces means the same thing in a LoRA run as it does here.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo06` | What are parameters, and does a bigger parameter count mean a better model? |
| `tn22` | What is the difference between an epoch, a step and a batch in LLM training? |

**Also asked in classical-ML rounds:** derive the MSE gradient and the normal equation; what does a learning rate that is too large do, and what is the largest stable one? why scale features for gradient descent? batch vs mini-batch vs SGD; what does R² mean, and can it be negative? what does a residual plot tell you?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#4a3aa7"])})

## 1. The model and the loss

**In plain English:** predict a number as a weighted sum of the inputs plus a constant. "Learning" means choosing the weights that make the average squared error on the training data as small as possible. The weights are the model's **parameters** (`fo06`); an LLM is the same idea with billions of them.

Stack a column of ones onto the features so the intercept is just another weight, $\theta = (b, w_1, \dots, w_d)$:

$$\hat y = X\theta \qquad \text{MSE}(\theta) = \frac{1}{n}\lVert X\theta - y\rVert^2 = \frac1n \sum_i (\hat y_i - y_i)^2$$

**Whiteboard derivation.** Expand $\frac1n (X\theta-y)^\top(X\theta-y)$ and differentiate:

$$\nabla_\theta\,\text{MSE} = \frac{2}{n}X^\top(X\theta - y)$$

Set it to zero and you get the **normal equations** $X^\top X\,\theta = X^\top y$, so $\theta^* = (X^\top X)^{-1}X^\top y$.

Why *squared* error? It is smooth and convex (one global minimum), it punishes big misses more than small ones, and it is the maximum-likelihood loss if the noise is Gaussian. Its weakness is outliers; that is what MAE and the Huber loss are for.

In [ ]:
n = 100
x = 2 * rng.random(n)
y = 4 + 3 * x + rng.normal(0, 1, n)              # truth: intercept 4, slope 3, noise sd 1
Xb = np.c_[np.ones(n), x]                         # column of ones -> the intercept is theta[0]

def mse(Xb, y, theta):
    r = Xb @ theta - y
    return r @ r / len(y)

def mse_grad(Xb, y, theta):
    return 2 / len(y) * Xb.T @ (Xb @ theta - y)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.scatter(x, y, s=14, color=BLUE, label="data")
ax.plot([0, 2], [4, 10], color=GREY, ls="--", label="true line y = 4 + 3x")
ax.set(title="Toy regression data", xlabel="x", ylabel="y"); ax.legend(); plt.show()

## 2. Closed form: the normal equation

Solve $X^\top X\,\theta = X^\top y$ with `np.linalg.solve` (never form the inverse explicitly; it is slower and less accurate). Libraries go one step further and use `lstsq` (a QR or SVD factorisation), which also copes when columns are nearly collinear.

In [ ]:
theta_closed = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)      # normal equation, solved rather than inverted
theta_lstsq, *_ = np.linalg.lstsq(Xb, y, rcond=None)     # what libraries do
print(f"normal equation: intercept {theta_closed[0]:.3f}, slope {theta_closed[1]:.3f}")
print(f"lstsq          : intercept {theta_lstsq[0]:.3f}, slope {theta_lstsq[1]:.3f}")
print(f"training MSE at the optimum: {mse(Xb, y, theta_closed):.3f}  (noise variance is 1.0)")
assert np.allclose(theta_closed, theta_lstsq)
assert abs(theta_closed[0] - 4) < 0.5 and abs(theta_closed[1] - 3) < 0.5   # close to the truth, not equal: noise

Cost: forming $X^\top X$ is $O(nd^2)$ and solving is $O(d^3)$. Great for thousands of features, hopeless for millions of parameters, and it only exists because this loss is quadratic. Neural networks have no closed form, so they need an iterative method.

## 3. Gradient descent from scratch

**In plain English:** stand on the loss surface, find the downhill direction (minus the gradient), take a small step, repeat. The step size is the **learning rate** $\eta$:

$$\theta \leftarrow \theta - \eta\,\nabla_\theta\,\text{MSE}(\theta)$$

First, an interview habit that catches most gradient bugs: check the analytic gradient against finite differences.

In [ ]:
theta_test, eps = np.array([1.0, -2.0]), 1e-6
numeric = np.array([(mse(Xb, y, theta_test + eps * e) - mse(Xb, y, theta_test - eps * e)) / (2 * eps)
                    for e in np.eye(2)])
analytic = mse_grad(Xb, y, theta_test)
print("numeric gradient :", numeric.round(5))
print("analytic gradient:", analytic.round(5))
assert np.allclose(numeric, analytic, rtol=1e-5)

In [ ]:
def gradient_descent(Xb, y, lr=0.1, epochs=200, theta0=(0.0, 0.0)):
    """Full-batch gradient descent. Returns final theta, the path of thetas, and the loss per step."""
    theta = np.array(theta0, dtype=float)
    path, losses = [theta.copy()], [mse(Xb, y, theta)]
    for _ in range(epochs):
        theta = theta - lr * mse_grad(Xb, y, theta)
        path.append(theta.copy())
        losses.append(mse(Xb, y, theta))
    return theta, np.array(path), np.array(losses)

theta_gd, path, losses = gradient_descent(Xb, y, lr=0.1, epochs=1000)
print(f"gradient descent: intercept {theta_gd[0]:.4f}, slope {theta_gd[1]:.4f}")
print(f"normal equation : intercept {theta_closed[0]:.4f}, slope {theta_closed[1]:.4f}")
assert np.allclose(theta_gd, theta_closed, atol=1e-3)     # both roads lead to the same minimum

## 4. The loss surface and the path gradient descent takes

MSE is a quadratic bowl over $(b, w)$. Each contour is a set of equal loss; the gradient is perpendicular to the contours, so gradient descent crosses them at right angles.

In [ ]:
B, Wg = np.meshgrid(np.linspace(-1, 8, 120), np.linspace(-1, 6, 120))
grid = np.c_[B.ravel(), Wg.ravel()]                          # every (b, w) pair on the grid
Z = ((Xb @ grid.T - y[:, None]) ** 2).mean(axis=0).reshape(B.shape)

fig, ax = plt.subplots(figsize=(7, 4.2))
cs = ax.contour(B, Wg, Z, levels=np.geomspace(Z.min() + 0.05, Z.max(), 14), cmap="Blues_r", linewidths=1)
ax.plot(path[:60, 0], path[:60, 1], "o-", color=ORANGE, ms=3, lw=1.2, label="gradient descent (lr=0.1), first 60 steps")
ax.plot(*theta_closed, "*", color="black", ms=12, label="closed-form minimum")
ax.set(title="MSE loss surface over (intercept, slope)", xlabel="intercept b", ylabel="slope w")
ax.legend(loc="upper right"); plt.show()

## 5. The learning rate: too small, about right, too large

For a quadratic loss the curvature is the Hessian $H = \frac2n X^\top X$ (a constant matrix). Gradient descent is stable only if $\eta < 2/\lambda_{\max}(H)$: past that, every step overshoots by more than it corrects. That is the one formula worth memorising here.

In [ ]:
H = 2 / n * Xb.T @ Xb
eigs = np.linalg.eigvalsh(H)
lr_max = 2 / eigs.max()
print(f"Hessian eigenvalues: {eigs.round(3)}   ->   largest stable learning rate 2/lambda_max = {lr_max:.3f}")

lrs = {"too small (lr=0.01)": 0.01, "about right (lr=0.3)": 0.3, f"too large (lr={1.05 * lr_max:.2f})": 1.05 * lr_max}
runs = {name: gradient_descent(Xb, y, lr=lr, epochs=80) for name, lr in lrs.items()}
loss_star = mse(Xb, y, theta_closed)

fig, ax = plt.subplots(figsize=(8, 3.6))
for name, (_, _, l) in runs.items():
    ax.plot(l - loss_star + 1e-12, lw=2, label=name)
ax.set_yscale("log")
ax.set(title="Loss curves for three learning rates", xlabel="epoch (= step, full batch)", ylabel="loss above the minimum (log)")
ax.legend(); plt.show()

(_, _, l_small), (_, _, l_ok), (_, _, l_big) = runs.values()
assert l_big[-1] > l_big[0]                      # too large: the loss grows
assert l_ok[-1] - loss_star < 1e-6               # about right: converged in 80 steps
assert l_small[-1] > l_ok[-1] + 0.01             # too small: still far away

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, (name, (_, p, _)) in zip(axes, runs.items()):
    ax.contour(B, Wg, Z, levels=np.geomspace(Z.min() + 0.05, Z.max(), 12), cmap="Blues_r", linewidths=0.8)
    p = p[:25]
    ax.plot(p[:, 0], p[:, 1], "o-", color=ORANGE, ms=3, lw=1)
    ax.plot(*theta_closed, "*", color="black", ms=10)
    ax.set(title=name, xlabel="intercept b", xlim=(-1, 8), ylim=(-1, 6))
axes[0].set_ylabel("slope w")
fig.suptitle("First 25 steps: crawl, converge, diverge", y=1.02); plt.show()

Read the shapes, because a real training run looks the same:

| Loss curve | Diagnosis | Fix |
|---|---|---|
| Falls, but painfully slowly | learning rate too small | raise it (try ×3 steps: 1e-4, 3e-4, 1e-3 ...) |
| Falls fast, then flat at a low value | about right | stop; or decay the rate to settle further |
| Zig-zags, or climbs to `inf`/`nan` | learning rate too large | lower it; add warm-up; check for unscaled features |

## 6. Feature scaling: why a long, thin valley is slow

**In plain English:** if one feature varies by ±10 and another by ±1, the bowl becomes a long, thin valley. The learning rate has to be small enough for the *steep* direction, so progress along the *shallow* direction crawls. Standardising (mean 0, standard deviation 1) makes the bowl round.

The number that captures it is the **condition number** $\kappa = \lambda_{\max}/\lambda_{\min}$ of the Hessian. Below, both runs use a learning rate of $1/\lambda_{\max}$ (safe for each).

In [ ]:
m = 500
F = np.c_[rng.normal(0, 1, m), rng.normal(0, 10, m)]      # feature 2 has 10x the spread of feature 1
yF = F @ np.array([2.0, 0.3]) + rng.normal(0, 0.5, m)
F, yF = F - F.mean(axis=0), yF - yF.mean()                # centred, so we can drop the intercept
Fs = F / F.std(axis=0)                                    # standardised copy

def gd_until(Xm, y, tol=1e-8, max_epochs=5000):
    """GD with lr = 1/lambda_max. Returns epochs needed, the loss-gap curve, the path and the condition number."""
    Hm = 2 / len(y) * Xm.T @ Xm
    ev = np.linalg.eigvalsh(Hm)
    theta_opt = np.linalg.lstsq(Xm, y, rcond=None)[0]
    best = mse(Xm, y, theta_opt)
    theta, gaps, path = np.zeros(Xm.shape[1]), [], []
    for epoch in range(max_epochs):
        path.append(theta.copy()); gaps.append(mse(Xm, y, theta) - best)
        if gaps[-1] < tol:
            break
        theta = theta - (1 / ev.max()) * mse_grad(Xm, y, theta)
    return epoch, np.array(gaps), np.array(path), ev.max() / ev.min()

ep_raw, gaps_raw, path_raw, kappa_raw = gd_until(F, yF)
ep_std, gaps_std, path_std, kappa_std = gd_until(Fs, yF)
print(f"raw features         : condition number {kappa_raw:6.1f}, epochs to converge {ep_raw}")
print(f"standardised features: condition number {kappa_std:6.1f}, epochs to converge {ep_std}")
assert ep_raw > 20 * ep_std

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
W1, W2 = np.meshgrid(np.linspace(-0.5, 3.0, 100), np.linspace(-0.1, 0.7, 100))
ZF = ((F @ np.c_[W1.ravel(), W2.ravel()].T - yF[:, None]) ** 2).mean(axis=0).reshape(W1.shape)
axes[0].contour(W1, W2, ZF, levels=np.geomspace(ZF.min() + 0.01, ZF.max(), 14), cmap="Blues_r", linewidths=0.8)
axes[0].plot(path_raw[:120, 0], path_raw[:120, 1], "o-", color=ORANGE, ms=2, lw=1)
axes[0].set(title="Raw features: a long thin valley (first 120 steps)", xlabel="w1 (feature sd 1)", ylabel="w2 (feature sd 10)")
axes[1].plot(gaps_raw + 1e-16, lw=2, label=f"raw (kappa={kappa_raw:.0f}): {ep_raw} epochs")
axes[1].plot(gaps_std + 1e-16, lw=2, label=f"standardised (kappa={kappa_std:.1f}): {ep_std} epochs")
axes[1].set_yscale("log")
axes[1].set(title="Loss above the minimum", xlabel="epoch", ylabel="loss gap (log)"); axes[1].legend()
plt.tight_layout(); plt.show()

Scaling changes nothing about *which* model is best (the fitted predictions are identical), only how fast gradient descent gets there. It matters for everything distance- or gradient-based: gradient descent, kNN, SVMs, k-means, PCA and any regularised model. Trees do not care. Deep learning bakes the same idea into the architecture with normalisation layers.

## 7. Batch vs mini-batch vs stochastic gradient descent

**In plain English:** *batch* GD looks at every row before each step (accurate but expensive). *Stochastic* GD looks at one row per step (cheap and noisy). *Mini-batch* uses a small handful (32 to 512); it is what everyone actually runs, on GPUs, for every neural network.

Vocabulary (`tn22`): a **batch** is the rows used for one update, a **step** is one update, an **epoch** is one pass over the data, so steps per epoch = ⌈n / batch size⌉. LLM training measures batches in tokens, and large pretraining runs barely complete one epoch.

In [ ]:
Xm, ym = make_regression(n_samples=1000, n_features=3, noise=10, random_state=0)
Xm = StandardScaler().fit_transform(Xm)
ym = (ym - ym.mean()) / ym.std()                          # scale the target too, so one lr suits all runs
Xmb = np.c_[np.ones(len(ym)), Xm]
loss_opt = mse(Xmb, ym, np.linalg.lstsq(Xmb, ym, rcond=None)[0])

def minibatch_gd(Xb, y, batch_size, lr=0.05, epochs=20, seed=0):
    """Shuffle every epoch, update once per batch; record the FULL-data loss after every step."""
    r = np.random.default_rng(seed)
    theta, losses = np.zeros(Xb.shape[1]), [mse(Xb, y, np.zeros(Xb.shape[1]))]
    for _ in range(epochs):
        order = r.permutation(len(y))
        for start in range(0, len(y), batch_size):
            idx = order[start:start + batch_size]
            theta = theta - lr * mse_grad(Xb[idx], y[idx], theta)
            losses.append(mse(Xb, y, theta))
    return theta, np.array(losses)

sizes = {"batch (1000 rows/step)": 1000, "mini-batch (32 rows/step)": 32, "SGD (1 row/step)": 1}
curves = {name: minibatch_gd(Xmb, ym, bs)[1] for name, bs in sizes.items()}
for name, bs in sizes.items():
    print(f"{name:>26}: {int(np.ceil(1000 / bs)):4d} steps per epoch, final loss gap {curves[name][-1] - loss_opt:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for name, bs in sizes.items():
    c = curves[name]
    steps_per_epoch = int(np.ceil(1000 / bs))
    ax.plot(np.arange(len(c)) / steps_per_epoch, c - loss_opt + 1e-12, lw=0.6 if bs == 1 else 2, alpha=0.8 if bs == 1 else 1, label=name)
ax.set_yscale("log"); ax.set_xlim(0, 20)
ax.set(title="Same data, same learning rate, different batch sizes", xlabel="epochs (passes over the data)",
       ylabel="full-data loss above the minimum (log)")
ax.legend(); plt.show()

gap = {name: c - loss_opt for name, c in curves.items()}
batch_c, mini_c, sgd_c = gap.values()
assert np.all(np.diff(curves["batch (1000 rows/step)"]) <= 1e-12)     # full-batch GD never goes uphill here
assert mini_c[1000 // 32] < batch_c[1]                                  # after one epoch mini-batch is far ahead
assert sgd_c[-1000:].std() > 10 * mini_c[-32:].std()                    # SGD keeps bouncing at the bottom

What the picture says:

| | Batch | Mini-batch | SGD |
|---|---|---|---|
| Rows per step | all *n* | 32 to 512 | 1 |
| Steps per epoch | 1 | n / batch | n |
| Gradient noise | none | some (a useful regulariser) | a lot |
| Hardware use | memory-bound for big *n* | fills a GPU efficiently | poor (no vectorisation) |
| Settles at the minimum? | yes | nearly (decay the rate to finish) | only with a decaying learning rate |

Per *epoch*, the small-batch runs make hundreds of updates while batch GD makes one, which is why nobody trains big models full-batch. The noise floor is why learning-rate schedules decay at the end ([deep learning notebook 02](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb)).

## 8. The same thing in scikit-learn

`LinearRegression` is the closed-form (least squares) solver; `SGDRegressor` is the gradient version for data too big for memory (it expects scaled features).

In [ ]:
lin = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"LinearRegression: intercept {lin.intercept_:.4f}, slope {lin.coef_[0]:.4f}")
print(f"ours (closed)   : intercept {theta_closed[0]:.4f}, slope {theta_closed[1]:.4f}")
assert np.allclose([lin.intercept_, lin.coef_[0]], theta_closed)

sgd = SGDRegressor(penalty=None, eta0=0.01, max_iter=1000, tol=1e-6, random_state=0).fit(Xm, ym)
theta_opt = np.linalg.lstsq(Xmb, ym, rcond=None)[0]
print(f"SGDRegressor coef: {sgd.coef_.round(4)}   least squares: {theta_opt[1:].round(4)}")
assert np.allclose(sgd.coef_, theta_opt[1:], atol=0.02)

## 9. Regression metrics: MSE, RMSE, MAE, R²

On a real dataset: `load_diabetes` (442 patients, 10 standardised features, target = disease progression one year later).

$$R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

R² compares your model to "always predict the mean": 1 is perfect, 0 is no better than the mean, and **on test data it can be negative**.

In [ ]:
Xd, yd = load_diabetes(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=0)
diab = LinearRegression().fit(Xd_tr, yd_tr)
pred = diab.predict(Xd_te)

err = pred - yd_te
ours = {"MSE": np.mean(err ** 2), "RMSE": np.sqrt(np.mean(err ** 2)), "MAE": np.mean(np.abs(err)),
        "R2": 1 - np.sum(err ** 2) / np.sum((yd_te - yd_te.mean()) ** 2)}
theirs = {"MSE": mean_squared_error(yd_te, pred), "RMSE": root_mean_squared_error(yd_te, pred),
          "MAE": mean_absolute_error(yd_te, pred), "R2": r2_score(yd_te, pred)}
for k in ours:
    print(f"{k:>4}: ours {ours[k]:9.3f}   sklearn {theirs[k]:9.3f}")
    assert np.isclose(ours[k], theirs[k])
print(f"target spread for scale: sd of y_test = {yd_te.std():.1f}")

r2_mean = r2_score(yd_te, np.full_like(yd_te, yd_tr.mean()))
r2_bad = r2_score(yd_te, diab.predict(Xd_te[::-1]))                 # predictions shuffled against the rows
print(f"R2 of 'predict the training mean': {r2_mean:.4f}   R2 of a broken model: {r2_bad:.3f}")
assert r2_mean < 0.02 and r2_bad < 0

| Metric | Units | Outliers | Use it when |
|---|---|---|---|
| MSE | target² | punished hard | it is your training loss; comparing models on one dataset |
| RMSE | target units | punished hard | reporting: "typically off by about RMSE" |
| MAE | target units | robust | big misses are not extra-bad; skewed targets |
| R² | none (≤ 1) | via MSE | comparing across targets; "how much variance explained" |

An R² of about a third is normal for noisy medical data like this. Whether it is *good enough* is a product question, not a statistics one.

## 10. Residual plots: what the model is missing

Plot residuals (actual − predicted) against predictions. A shapeless band around zero is what you want. A curve means a missing feature; a funnel means the error grows with the prediction.

In [ ]:
xc = rng.uniform(-3, 3, 200)
yc = 0.5 * xc ** 2 + xc + 2 + rng.normal(0, 0.5, 200)          # truly curved
line_fit = LinearRegression().fit(xc.reshape(-1, 1), yc)
quad_X = np.c_[xc, xc ** 2]
quad_fit = LinearRegression().fit(quad_X, yc)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
axes[0].scatter(pred, yd_te - pred, s=12, color=BLUE)
axes[0].set(title="Diabetes: shapeless band (fine)", xlabel="predicted", ylabel="residual")
axes[1].scatter(line_fit.predict(xc.reshape(-1, 1)), yc - line_fit.predict(xc.reshape(-1, 1)), s=12, color=ORANGE)
axes[1].set(title="Straight line on curved data: U-shape", xlabel="predicted")
axes[2].scatter(quad_fit.predict(quad_X), yc - quad_fit.predict(quad_X), s=12, color=AQUA)
axes[2].set(title="After adding x² as a feature", xlabel="predicted")
for ax in axes:
    ax.axhline(0, color=GREY, lw=1)
plt.tight_layout(); plt.show()

r2_line, r2_quad = line_fit.score(xc.reshape(-1, 1), yc), quad_fit.score(quad_X, yc)
print(f"R2 straight line: {r2_line:.3f}   R2 with x² feature: {r2_quad:.3f}")
assert r2_quad > r2_line + 0.2

"Linear" means linear **in the parameters**, not in the inputs: adding $x^2$ as a column is still linear regression. Push that idea too far (degree 15) and you overfit, which is [notebook 05](05_overfitting_regularization_cross_validation.ipynb).

The textbook assumptions, and which ones matter for what:

| Assumption | Broken looks like | Matters for |
|---|---|---|
| Linear in parameters | curved residuals | predictions |
| Independent errors | patterns over time or group | honest error bars |
| Constant variance | funnel-shaped residuals | honest error bars; consider log(y) |
| Normal errors | skewed residual histogram | p-values and intervals only |
| No strong collinearity | wild, sign-flipping coefficients | interpreting coefficients; Ridge fixes it |

## Try it yourself

**1.** Write gradient descent with a *separate* intercept `b` and slope `w` (no column of ones): derive both partial derivatives, run it on `x, y`, and assert you land on `theta_closed`.

In [ ]:
# Solution — try it yourself first, then run this
w, b, lr = 0.0, 0.0, 0.1
for _ in range(2000):
    err = w * x + b - y
    w -= lr * 2 * np.mean(err * x)      # dMSE/dw = (2/n) * sum(err * x)
    b -= lr * 2 * np.mean(err)          # dMSE/db = (2/n) * sum(err)
print(f"b = {b:.4f}, w = {w:.4f}")
assert np.allclose([b, w], theta_closed, atol=1e-3)

**2.** For the diabetes training set (with a column of ones), compute the largest stable learning rate $2/\lambda_{\max}$. Verify it: run 300 epochs at 0.95× and at 1.05× that rate and show that one converges and the other blows up.

In [ ]:
# Solution — try it yourself first, then run this
Xd_b = np.c_[np.ones(len(yd_tr)), Xd_tr]
lr_limit = 2 / np.linalg.eigvalsh(2 / len(yd_tr) * Xd_b.T @ Xd_b).max()
best_mse = mse(Xd_b, yd_tr, np.linalg.lstsq(Xd_b, yd_tr, rcond=None)[0])
print(f"largest stable learning rate: {lr_limit:.3f}   (least-squares training MSE: {best_mse:.0f})")
results = {}
for factor in (0.95, 1.05):
    theta = np.zeros(Xd_b.shape[1])
    for _ in range(300):
        theta = theta - factor * lr_limit * mse_grad(Xd_b, yd_tr, theta)
    results[factor] = mse(Xd_b, yd_tr, theta)
    print(f"lr = {factor:.2f} x limit -> training MSE after 300 epochs: {results[factor]:.3g}")
assert results[1.05] > 100 * results[0.95]
# 0.95x is stable but still creeping: load_diabetes columns are scaled to unit *norm*, not unit variance,
# so the intercept dominates the curvature (a badly conditioned problem, exactly section 6)

**3.** Outliers. Add five wild points to the toy data. Fit a line by minimising MSE (closed form) and by minimising **MAE** with *sub*gradient descent (the gradient of |r| is sign(r); use a decaying learning rate). Which slope stays near the true 3?

In [ ]:
# Solution — try it yourself first, then run this
x_out = np.r_[x, [1.8, 1.9, 1.95, 2.0, 1.85]]
y_out = np.r_[y, [-10, -12, -9, -11, -10]]                    # five wild points at the right edge
Xo = np.c_[np.ones(len(x_out)), x_out]
theta_mse = np.linalg.lstsq(Xo, y_out, rcond=None)[0]
theta_mae = np.zeros(2)
for t in range(5000):
    theta_mae -= (0.5 / np.sqrt(t + 1)) * Xo.T @ np.sign(Xo @ theta_mae - y_out) / len(y_out)
print(f"MSE fit slope: {theta_mse[1]:.2f}   MAE fit slope: {theta_mae[1]:.2f}   (truth 3)")
assert abs(theta_mae[1] - 3) < abs(theta_mse[1] - 3)

## Say it in an interview

- **30-second answer:** "Linear regression predicts ŷ = Xθ and minimises mean squared error. The gradient is (2/n)Xᵀ(Xθ − y). Setting it to zero gives the normal equation θ = (XᵀX)⁻¹Xᵀy, exact and O(nd² + d³), fine for modest d. Gradient descent repeats θ ← θ − η·gradient; it scales to huge data and is how every neural network trains."
- **Learning rate:** too small crawls, too large diverges. For a quadratic loss the ceiling is η < 2/λ_max of the Hessian. In practice: sweep on a log scale and read the loss curve.
- **Scaling:** unscaled features give a badly conditioned, long-valley loss surface. Standardise for GD, kNN, SVM, k-means, PCA and regularised models; trees do not care.
- **Batch sizes:** batch = exact but one step per epoch; SGD = cheap, noisy, never settles without decay; mini-batch is the default. Epoch = one pass, step = one update (`tn22`).
- **Metrics:** RMSE in target units; MAE is robust to outliers; R² is relative to predicting the mean and can be negative on test data.
- **Traps:** inverting XᵀX explicitly; forgetting the intercept; reporting train R²; ignoring a curved residual plot; reading coefficients of collinear features as causal; extrapolating outside the training range.
- **Follow-ups:** "Why not always use the closed form?" (d³ cost, no closed form for anything non-linear). "Why is MSE convex here but a neural network's loss is not?" (the model is linear in θ; stack non-linear layers and the bowl turns into a landscape).

## Next

- [03 · Logistic regression](03_logistic_regression_classification.ipynb): the same loop for classification, with a different loss.
- [05 · Overfitting, regularisation and cross-validation](05_overfitting_regularization_cross_validation.ipynb): polynomial features, Ridge and Lasso.
- [A neuron and backprop from scratch](../06_deep_learning/01_neuron_and_backprop_from_scratch.ipynb) and [training loops, optimisers and LR schedules](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb): gradient descent at scale.
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapters 3 and 4), [dl_fundamentals course](../../dl_fundamentals/index.html), [interview bank](../../ai_interview_prep/index.html).